# 02 — Data Validation

**Day 1, Step 2.** Define what 'valid' means *before* cleaning, so the cleaning
step has a specification to satisfy rather than a vibe.

The Build Notes suggest plain pandas asserts first and Pandera later. We go
straight to Pandera because the document already names it as the destination —
the rules then live in one place instead of scattered across notebooks, and the
same objects are imported by the pipeline, the API and the tests.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.validation.runner import validate
from hrai.validation.schemas import attrition_raw_schema, engagement_raw_schema

att = load_raw("employee_attrition")
eng = load_raw("hr_performance_engagement")

2026-08-28 01:54:13 | INFO  | raw dataset loaded


2026-08-28 01:54:13 | INFO  | raw dataset loaded


## Population A is genuinely clean

In [3]:
report = validate(att, attrition_raw_schema(), "employee_attrition", mode="report")
print(report.summary())

2026-08-28 01:54:13 | INFO  | schema validation passed


employee_attrition                 PASS


## Population B is not

Three real defects, all of which the cleaning step must fix.

In [4]:
report = validate(eng, engagement_raw_schema(), "hr_performance_engagement", mode="report")
print(report.summary())
pd.DataFrame(report.failures)

2026-08-28 01:54:13 | WARNING | schema validation failed


hr_performance_engagement          FAIL (1198 violations)


,check,column,scope,count,example
0,ExitDate is populated but EmployeeStatus is no...,None,dataframe,1198,6


### Finding F6 — the scale in the Build Notes is wrong for this data

Step 2 of the Build Notes specifies an engagement range check of **0–100**.
This file is on a **1–5 Likert scale**. Applied literally, that rule would pass
every single row and catch nothing — the most dangerous kind of validation,
because it looks like coverage.

In [5]:
likert = ["Engagement Score", "Satisfaction Score",
          "Work-Life Balance Score", "Current Employee Rating"]
eng[likert].describe().loc[["min", "max", "mean"]].round(2)

,Engagement Score,Satisfaction Score,Work-Life Balance Score,Current Employee Rating
min,1.00,1.00,1.00,1.00
max,5.00,5.00,5.00,5.00
mean,2.94,3.02,2.99,2.97


### The ExitDate contradiction

1,198 rows carry an `ExitDate` while `EmployeeStatus` says Active, Future Start
or Leave of Absence. Which column is telling the truth?

In [6]:
has_exit = eng["ExitDate"].notna() & eng["ExitDate"].astype(str).str.strip().ne("")
print(f"rows with an ExitDate: {has_exit.sum():,}\n")
print(pd.crosstab(eng["EmployeeStatus"], eng["TerminationType"]))

rows with an ExitDate: 1,606

TerminationType         Involuntary  Resignation  Retirement   Unk  Voluntary
EmployeeStatus                                                               
Active                          254          264         255  1544        266
Future Start                     19           11          23     0         17
Leave of Absence                 20           23          24     0         22
Terminated for Cause             23           22          10     0         14
Voluntarily Terminated           91           77          81     0         90


`TerminationType` is distributed almost uniformly *within* every
`EmployeeStatus` — `Voluntarily Terminated` employees split 91/77/81/90 across
Involuntary/Resignation/Retirement/Voluntary. That is noise, not signal. And
`ExitDate` is populated even for `Future Start` employees, who by definition
cannot have left.

**Decision: `EmployeeStatus` is authoritative.** `ExitDate`, `TerminationType`
and `TerminationDescription` are dropped as unreliable — which also satisfies
the leakage register (F7), since they *are* the label.

In [7]:
print("Leakage register — columns that ARE the label:")
for dataset, columns in get("leakage").items():
    print(f"  {dataset}: {columns}")

Leakage register — columns that ARE the label:
  hr_performance_engagement: ['ExitDate', 'TerminationType', 'TerminationDescription', 'EmployeeStatus']
  employee_attrition: ['Attrition']
  never_features: ['EmployeeNumber', 'Employee ID']
